# LeetCode 2261 — K Divisible Elements Subarrays bằng Python

Link đề: https://leetcode.com/problems/k-divisible-elements-subarrays/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Trie, Rolling Hash, Hash Function, Enumeration

Skeleton LeetCode:

```python
class Solution:
    def countDistinct(self, nums: List[int], k: int, p: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Liệt kê mọi **mảng con liên tiếp** bằng hai vòng lặp và duy trì **bộ đếm tăng dần** (số phần tử chia hết cho `p`) để dừng sớm.
2. Đếm mảng con **phân biệt theo nội dung**: dùng `set` các `tuple`, hiểu chi phí O(độ dài) khi băm `tuple`.
3. Đạt O(n²) bằng **cây tiền tố (trie)**: mỗi điểm đầu chèn dần các phần tử; mỗi nút mới = một mảng con mới.
4. Đạt O(n²) bằng **băm lăn (rolling hash)**, và hiểu rủi ro **va chạm** (collision) — vì sao dùng băm kép.
5. Tránh các bẫy: nối số thành chuỗi **không có dấu phân cách**, đếm trùng, nhầm điều kiện chia hết.

## 2. Đề bài & ví dụ

Cho mảng `nums` và hai số `k`, `p`. Trả về số **mảng con phân biệt** có **tối đa `k`** phần tử chia hết cho `p`.

Hai mảng **khác nhau** nếu khác độ dài, hoặc khác ở ít nhất một vị trí. (Hai mảng con ở hai vị trí khác nhau nhưng cùng nội dung chỉ tính **một** lần.)

**Ví dụ 1:** `nums = [2,3,3,2,2], k = 2, p = 2` → `11`: `[2], [2,3], [2,3,3], [2,3,3,2], [3], [3,3], [3,3,2], [3,3,2,2], [3,2], [3,2,2], [2,2]`. Mảng `[2,3,3,2,2]` có 3 số chẵn nên bị loại.

**Ví dụ 2:** `nums = [1,2,3,4], k = 4, p = 1` → `10` (mọi mảng con đều hợp lệ và đều khác nhau).

**Ràng buộc:** `1 ≤ n ≤ 200`, `1 ≤ nums[i], p ≤ 200`, `1 ≤ k ≤ n`. *Follow up:* làm được O(n²) không?

In [1]:
from typing import List
import random, time

examples = [([2, 3, 3, 2, 2], 2, 2, 11), ([1, 2, 3, 4], 4, 1, 10)]
for nums, k, p, exp in examples:
    print(nums, f"k={k}, p={p}", "-> kỳ vọng", exp, "| phần tử chia hết cho p:", [x for x in nums if x % p == 0])

[2, 3, 3, 2, 2] k=2, p=2 -> kỳ vọng 11 | phần tử chia hết cho p: [2, 2, 2]
[1, 2, 3, 4] k=4, p=1 -> kỳ vọng 10 | phần tử chia hết cho p: [1, 2, 3, 4]


## 3. Kiến thức nền

### 3.1 Liệt kê mảng con bằng hai vòng lặp

Mảng con `nums[i..j]` với `0 ≤ i ≤ j < n`; có `n(n+1)/2` mảng con (20 100 với n = 200).

In [2]:
a = [2, 3, 3]
print([a[i:j + 1] for i in range(len(a)) for j in range(i, len(a))])
print("n=200 có", 200 * 201 // 2, "mảng con")

[[2], [2, 3], [2, 3, 3], [3], [3, 3], [3]]
n=200 có 20100 mảng con


### 3.2 `tuple` làm khóa của `set` (băm theo nội dung)

`list` không băm được; `tuple(a[i:j+1])` thì được, và hai tuple cùng nội dung có cùng giá trị băm ⇒ `set` tự khử trùng. Nhưng tạo và băm tuple dài `L` tốn **O(L)**.

In [3]:
s = {(2,), (3,), (2,), (2, 3)}
print(s, len(s))

{(2, 3), (2,), (3,)} 3


### 3.3 Cây tiền tố (trie) bằng `dict` lồng nhau

Mỗi nút là một `dict` *giá trị con → nút con*. Chèn dãy `x₀, x₁, …` từ gốc: đi xuống, tạo nút khi chưa có. **Mỗi nút (trừ gốc) ứng với đúng một dãy tiền tố** ⇒ số nút = số dãy phân biệt đã chèn (tính cả các tiền tố).

In [4]:
root = {}
def insert(seq):
    node = root; created = 0
    for x in seq:
        if x not in node:
            node[x] = {}; created += 1
        node = node[x]
    return created
print(insert([2, 3, 3]), insert([2, 3]), insert([2, 4]), "-> cây:", root)

3 0 1 -> cây: {2: {3: {3: {}}, 4: {}}}


### 3.4 Băm lăn đa thức

`h(x₀..x_j) = (h(x₀..x_{j−1}) · B + x_j) mod P`. Thêm một phần tử vào cuối ⇒ O(1). Hai dãy khác nhau **có thể** trùng giá trị băm (va chạm) — xác suất nhỏ nếu `P` lớn, giảm thêm bằng **băm kép** (hai cặp `(B, P)` độc lập).

In [5]:
def poly_hash(seq, B=131, P=10**9 + 7):
    h = 0
    for x in seq: h = (h * B + x) % P
    return h
print(poly_hash([2, 3]), poly_hash([3, 2]), poly_hash([2, 3, 3]))

265 395 34718


## 4. Ý tưởng nền tảng / trực giác

Hai việc tách biệt:
1. **Lọc** mảng con hợp lệ: số phần tử chia hết cho `p` ≤ `k`.
2. **Khử trùng** theo nội dung.

**Việc 1 — dừng sớm:** cố định điểm đầu `i`, mở rộng `j`; bộ đếm `cnt` chỉ tăng ⇒ khi `cnt > k` thì mọi `j` lớn hơn cũng vi phạm ⇒ `break`.

**Việc 2 — khử trùng:** các mảng con cùng điểm đầu `i` là các **tiền tố** của hậu tố `nums[i:]` ⇒ chèn hậu tố vào trie từng phần tử một; một mảng con mới xuất hiện ⇔ tạo nút mới.

Làm tay ví dụ 1 theo điểm đầu (`p = 2`, `k = 2`):

| i | mảng con hợp lệ theo thứ tự j tăng | mới? |
|---|---|---|
| 0 | [2], [2,3], [2,3,3], [2,3,3,2] (dừng: [2,3,3,2,2] có 3 số chẵn) | 4 mới |
| 1 | [3], [3,3], [3,3,2], [3,3,2,2] | 4 mới |
| 2 | [3] (trùng), [3,2], [3,2,2] | 2 mới |
| 3 | [2] (trùng), [2,2] | 1 mới |
| 4 | [2] (trùng) | 0 |

Tổng 4 + 4 + 2 + 1 = 11. ✔

In [6]:
nums, k, p, _ = examples[0]
seen = set()
for i in range(len(nums)):
    cnt = 0; row = []
    for j in range(i, len(nums)):
        cnt += nums[j] % p == 0
        if cnt > k: row.append("DỪNG"); break
        t = tuple(nums[i:j + 1]); row.append(f"{list(t)}{'' if t not in seen else '(trùng)'}"); seen.add(t)
    print(f"i={i}: " + ", ".join(row))
print("tổng phân biệt:", len(seen))

i=0: [2], [2, 3], [2, 3, 3], [2, 3, 3, 2], DỪNG
i=1: [3], [3, 3], [3, 3, 2], [3, 3, 2, 2]
i=2: [3](trùng), [3, 2], [3, 2, 2]
i=3: [2](trùng), [2, 2]
i=4: [2](trùng)
tổng phân biệt: 11


## 5. Cách 1 — Brute Force / baseline: mọi mảng con, đếm lại số chia hết, gom `set`

In [7]:
def brute(nums, k, p):
    n = len(nums); s = set()
    for i in range(n):
        for j in range(i, n):
            sub = nums[i:j + 1]
            if sum(1 for x in sub if x % p == 0) <= k:
                s.add(tuple(sub))
    return len(s)

for nums, k, p, exp in examples:
    got = brute(nums, k, p)
    print(got, "OK" if got == exp else "SAI")

11 OK
10 OK


**Độ phức tạp:** O(n²) mảng con × O(n) đếm + O(n) tạo tuple ⇒ **O(n³)** = 8·10⁶ với n = 200 — vẫn chạy được trên LeetCode, nhưng không đạt follow-up O(n²).

**Nút thắt:** (1) đếm lại số chia hết từ đầu cho mỗi `j`; (2) mỗi mảng con tạo một tuple mới O(độ dài) — dù tuple của `(i, j)` chỉ là tuple của `(i, j−1)` thêm một phần tử.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — bộ đếm chạy + dừng sớm:** `cnt(i, j) = cnt(i, j−1) + [nums[j] % p == 0]`; dừng khi `cnt > k`.

**Insight 2 — trie biến "khử trùng" thành O(1) mỗi bước:** với điểm đầu `i`, đi từ gốc trie theo `nums[i], nums[i+1], …`. Bước `j`: nếu con `nums[j]` của nút hiện tại **chưa có** ⇒ mảng con `nums[i..j]` chưa từng gặp ⇒ tạo nút, `ans += 1`. Mỗi bước O(1) (tra `dict`) ⇒ tổng **O(n²)** thời gian và O(n²) nút.

**Insight 3 — băm lăn:** `h(i, j) = h(i, j−1)·B + nums[j]` ⇒ mã băm của mọi mảng con tính trong O(n²); cho vào `set` các số nguyên (băm số nguyên O(1)). Kết hợp **độ dài** hoặc **băm kép** để giảm va chạm.

**Quy tắc:** **nếu** `cnt > k` **thì** `break` (không cần xét `j` lớn hơn); **nếu** nút con chưa có **thì** đếm mảng con mới.

**Vì sao trie chính xác 100%?** Nút trong trie tương ứng **một-một** với dãy (đường đi từ gốc), không có va chạm như băm.

### 6.1 Chi phí ẩn của `set(tuple)`

Thêm `tuple` dài `L` vào `set` cần **tạo** tuple (O(L)) và **băm** nó (O(L)). Tổng trên mọi mảng con là `Σ L ≈ n³/6` ≈ 1,3·10⁶ với n = 200 — chạy được, nhưng tăng gấp ~70 lần so với O(n²) = 2·10⁴ bước của trie. Đo trực tiếp số "đơn vị công việc":

In [8]:
n = 200
work_tuple = sum(j - i + 1 for i in range(n) for j in range(i, n))
work_trie = n * (n + 1) // 2
print(f"tổng độ dài các tuple: {work_tuple:,} | số bước trie: {work_trie:,} | tỉ lệ ≈ {work_tuple / work_trie:.0f}")

tổng độ dài các tuple: 1,353,400 | số bước trie: 20,100 | tỉ lệ ≈ 67


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Bộ đếm chạy + dừng sớm + `set` các tuple

In [9]:
def running_tuples(nums, k, p):
    n = len(nums); s = set()
    for i in range(n):
        cnt = 0
        for j in range(i, n):
            cnt += nums[j] % p == 0
            if cnt > k: break
            s.add(tuple(nums[i:j + 1]))
    return len(s)

for nums, k, p, exp in examples:
    got = running_tuples(nums, k, p)
    print(got, "OK" if got == exp else "SAI")

11 OK
10 OK


Vẫn O(n³) worst (do tạo tuple), nhưng nhanh hơn cách 1 nhờ bỏ vòng đếm và dừng sớm.

### 7.2 Cách 3 — Trie (O(n²), chính xác)

In [10]:
def trie_count(nums, k, p):
    n = len(nums); root = {}; ans = 0
    for i in range(n):
        node = root; cnt = 0
        for j in range(i, n):
            x = nums[j]
            cnt += x % p == 0
            if cnt > k: break
            nxt = node.get(x)
            if nxt is None:
                nxt = {}; node[x] = nxt; ans += 1       # mảng con nums[i..j] mới
            node = nxt
    return ans

for nums, k, p, exp in examples:
    got = trie_count(nums, k, p)
    print(got, "OK" if got == exp else "SAI")

11 OK
10 OK


**Độ phức tạp cách 3:** O(n²) thời gian, O(n²) bộ nhớ (≤ 20 100 nút).

### 7.3 Cách 4 — Băm lăn kép (O(n²), xác suất đúng rất cao)

In [11]:
def double_hash(nums, k, p):
    n = len(nums); s = set()
    B1, B2, P1, P2 = 211, 13331, 10**9 + 7, 10**9 + 9
    for i in range(n):
        h1 = h2 = cnt = 0
        for j in range(i, n):
            cnt += nums[j] % p == 0
            if cnt > k: break
            h1 = (h1 * B1 + nums[j]) % P1
            h2 = (h2 * B2 + nums[j]) % P2
            s.add(h1 << 32 | h2)              # gộp hai mã băm thành một số nguyên
    return len(s)

for nums, k, p, exp in examples:
    got = double_hash(nums, k, p)
    print(got, "OK" if got == exp else "SAI")

11 OK
10 OK


Chọn `B > max(nums)` (ở đây `B₁ = 211 > 200`) để tránh va chạm "cấu trúc" kiểu `1·B + 23 = 3·B + 3` khi `B = 10` (xem Lỗi 4); `P` là số nguyên tố lớn để xác suất va chạm ngẫu nhiên nhỏ; hai cặp `(B, P)` độc lập làm xác suất đó nhỏ hơn nữa. Gộp hai mã thành một số nguyên `h1 << 32 | h2` (vì `h2 < 2³²`) để `set` băm nhanh.

**Độ phức tạp cách 4:** O(n²) thời gian, O(n²) bộ nhớ (số nguyên).

### 7.4 Cách 5 — Python: băm lăn bằng **số nguyên lớn chính xác** (không modulo)

Python có số nguyên tùy ý: `h = h · B + x` với `B = 201 > max(nums)` là **biểu diễn cơ số B** của dãy — **không bao giờ va chạm** giữa hai dãy cùng độ dài; kèm độ dài để phân biệt dãy khác độ dài. Chi phí: phép nhân số lớn O(L/64) — với n = 200, số có ≤ ~1600 bit, vẫn nhanh.

In [12]:
def exact_bigint(nums, k, p):
    n = len(nums); s = set(); B = 201
    for i in range(n):
        h = cnt = 0
        for j in range(i, n):
            cnt += nums[j] % p == 0
            if cnt > k: break
            h = h * B + nums[j]
            s.add((j - i + 1, h))            # (độ dài, giá trị cơ số B) là duy nhất
    return len(s)

for nums, k, p, exp in examples:
    print(exact_bigint(nums, k, p), "OK" if exact_bigint(nums, k, p) == exp else "SAI")

11 OK
10 OK


Vì sao cần độ dài? Với giá trị ≥ 1, `[x]` và `[0, x]` sẽ trùng nếu có số 0 — ở đây `nums[i] ≥ 1` nên thực ra không cần, nhưng thêm độ dài là thói quen tốt.

## 8. TRACE từng bước

### 8.1 Trace cách 3 (trie) trên ví dụ 1

In [13]:
def trace_trie(nums, k, p):
    n = len(nums); root = {}; ans = 0
    for i in range(n):
        node = root; cnt = 0; log = []
        for j in range(i, n):
            x = nums[j]; cnt += x % p == 0
            if cnt > k: log.append(f"j={j}: cnt={cnt}>k DỪNG"); break
            if x not in node:
                node[x] = {}; ans += 1; log.append(f"j={j}: {nums[i:j+1]} MỚI")
            else:
                log.append(f"j={j}: {nums[i:j+1]} đã có")
            node = node[x]
        print(f"i={i}: " + " | ".join(log) + f"  => ans={ans}")
    return ans

trace_trie([2, 3, 3, 2, 2], 2, 2)

i=0: j=0: [2] MỚI | j=1: [2, 3] MỚI | j=2: [2, 3, 3] MỚI | j=3: [2, 3, 3, 2] MỚI | j=4: cnt=3>k DỪNG  => ans=4
i=1: j=1: [3] MỚI | j=2: [3, 3] MỚI | j=3: [3, 3, 2] MỚI | j=4: [3, 3, 2, 2] MỚI  => ans=8
i=2: j=2: [3] đã có | j=3: [3, 2] MỚI | j=4: [3, 2, 2] MỚI  => ans=10
i=3: j=3: [2] đã có | j=4: [2, 2] MỚI  => ans=11
i=4: j=4: [2] đã có  => ans=11


11

### 8.2 Đếm số nút trie (= đáp án) và in cấu trúc

In [14]:
def build_trie(nums, k, p):
    root = {}
    for i in range(len(nums)):
        node = root; cnt = 0
        for j in range(i, len(nums)):
            cnt += nums[j] % p == 0
            if cnt > k: break
            node = node.setdefault(nums[j], {})
    return root

def show_trie(node, prefix="", depth=0):
    for x, child in sorted(node.items()):
        print("  " * depth + f"└ {x}")
        show_trie(child, prefix, depth + 1)

def count_nodes(node):
    return sum(1 + count_nodes(c) for c in node.values())

t = build_trie([2, 3, 3, 2, 2], 2, 2)
show_trie(t); print("số nút =", count_nodes(t))

└ 2
  └ 2
  └ 3
    └ 3
      └ 2
└ 3
  └ 2
    └ 2
  └ 3
    └ 2
      └ 2
số nút = 11


## 9. Code nộp LeetCode

Chọn trie (chính xác, O(n²)).

In [15]:
from typing import List

class Solution:
    def countDistinct(self, nums: List[int], k: int, p: int) -> int:
        root = {}
        ans = 0
        n = len(nums)
        for i in range(n):
            node = root
            cnt = 0
            for j in range(i, n):
                x = nums[j]
                if x % p == 0:
                    cnt += 1
                    if cnt > k:
                        break                     # mọi j lớn hơn cũng vi phạm
                nxt = node.get(x)
                if nxt is None:                   # mảng con nums[i..j] lần đầu xuất hiện
                    nxt = {}
                    node[x] = nxt
                    ans += 1
                node = nxt
        return ans

In [16]:
sol = Solution()
for nums, k, p, exp in examples:
    got = sol.countDistinct(nums, k, p)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

11 | kỳ vọng 11 | OK
10 | kỳ vọng 10 | OK


## 10. Vì sao thuật toán đúng

**Song ánh nút ↔ dãy.** Trong trie, mỗi nút `v ≠ gốc` ứng với đúng một dãy (nhãn trên đường đi từ gốc tới `v`), và hai nút khác nhau ứng với hai dãy khác nhau.

**Mọi mảng con hợp lệ đều được chèn.** Với mảng con hợp lệ `nums[i..j]` (tức `cnt(i, j) ≤ k`), vì `cnt` không giảm theo `j`, mọi `nums[i..j']` với `j' ≤ j` cũng hợp lệ ⇒ vòng lặp của điểm đầu `i` không `break` trước `j` ⇒ đường đi `nums[i], …, nums[j]` được tạo trong trie.

**Không có dãy không hợp lệ nào được chèn.** Ta `break` ngay khi `cnt > k`, trước khi chèn.

**Đếm đúng.** `ans` tăng đúng một khi tạo một nút ⇒ `ans` = số nút = số dãy phân biệt đã chèn = số mảng con hợp lệ phân biệt (mỗi dãy được chèn ít nhất một lần, các lần sau không tạo nút). ∎

**Băm (cách 4):** đúng nếu không có va chạm; với băm kép mod hai số nguyên tố ~10⁹, xác suất va chạm giữa ~2·10⁴ dãy là cỡ `(2·10⁴)² / 10¹⁸ ≈ 4·10⁻¹⁰` — gần như không thể. Cách 5 (số lớn chính xác) đúng tuyệt đối.

## 11. Phân tích Time / Space Complexity

`n ≤ 200`.

| Cách | Time | Space |
|---|---|---|
| 1. Mọi mảng con, đếm lại | O(n³) | O(n³) (tuple) |
| 2. Đếm chạy + tuple | O(n³) worst | O(n³) |
| 3. Trie | O(n²) | O(n²) nút |
| 4. Băm lăn kép | O(n²) | O(n²) số nguyên |
| 5. Số lớn chính xác | O(n² · n/64) | O(n²) số lớn |

In [17]:
rng = random.Random(2261)
big = [rng.randint(1, 200) for _ in range(200)]
dup = [rng.randint(1, 3) for _ in range(200)]          # nhiều mảng con trùng
for data, name_d in [(big, "ngẫu nhiên"), (dup, "nhiều trùng")]:
    res = []
    for name, f in [("brute", brute), ("running_tuples", running_tuples), ("trie_count", trie_count),
                    ("double_hash", double_hash), ("exact_bigint", exact_bigint), ("Solution", Solution().countDistinct)]:
        t = time.perf_counter(); res.append(f(data, 200, 1)); dt = time.perf_counter() - t
        print(f"{name_d:<11} {name:<15} -> {res[-1]}  {dt*1000:6.0f} ms")
    assert len(set(res)) == 1

ngẫu nhiên  brute           -> 20024      70 ms
ngẫu nhiên  running_tuples  -> 20024      23 ms
ngẫu nhiên  trie_count      -> 20024       4 ms
ngẫu nhiên  double_hash     -> 20024       5 ms
ngẫu nhiên  exact_bigint    -> 20024       5 ms
ngẫu nhiên  Solution        -> 20024       3 ms


nhiều trùng brute           -> 19308      65 ms


nhiều trùng running_tuples  -> 19308      20 ms
nhiều trùng trie_count      -> 19308       4 ms
nhiều trùng double_hash     -> 19308       4 ms
nhiều trùng exact_bigint    -> 19308       5 ms


nhiều trùng Solution        -> 19308      25 ms


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [18]:
ALL = {"brute": brute, "running_tuples": running_tuples, "trie_count": trie_count,
       "double_hash": double_hash, "exact_bigint": exact_bigint, "Solution": Solution().countDistinct}
edge = examples + [
    ([5], 1, 5, 1), ([5], 1, 2, 1),
    ([5, 5, 5], 3, 1, 3),                 # [5], [5,5], [5,5,5]
    ([5, 5, 5], 1, 5, 1),                 # chỉ [5]
    ([1, 1, 1], 1, 2, 3),                 # không số nào chia hết cho 2: [1], [1,1], [1,1,1]
    ([2, 4, 6], 1, 2, 3),                 # chỉ các mảng con độ dài 1
    ([1, 2, 1, 2], 4, 3, 7),              # [1],[2],[1,2],[2,1],[1,2,1],[2,1,2],[1,2,1,2]
]
for nums, k, p, exp in edge:
    for name, f in ALL.items():
        assert f(list(nums), k, p) == exp, (name, nums, k, p, f(nums, k, p), exp)
print("Edge case OK:", len(edge))

Edge case OK: 9


### 12.2 Đối chiếu ngẫu nhiên (seed cố định)

In [19]:
rng = random.Random(42)
for _ in range(800):
    n = rng.randint(1, 25); hi = rng.choice([2, 3, 5, 200])
    nums = [rng.randint(1, hi) for _ in range(n)]
    k = rng.randint(1, n); p = rng.randint(1, 6)
    ref = brute(nums, k, p)
    for name, f in ALL.items():
        assert f(list(nums), k, p) == ref, (name, nums, k, p)
print("800 case ngẫu nhiên OK")

800 case ngẫu nhiên OK


### 12.3 Tính chất: tăng `k` không làm giảm đáp án; 100 số lẻ phân biệt với `p = 2` ⇒ mọi mảng con hợp lệ và khác nhau ⇒ `100·101/2`

In [20]:
rng = random.Random(7)
for _ in range(200):
    nums = [rng.randint(1, 6) for _ in range(rng.randint(1, 20))]; p = rng.randint(1, 4)
    vals = [Solution().countDistinct(nums, k, p) for k in range(1, len(nums) + 1)]
    assert vals == sorted(vals)
odd = list(range(1, 200, 2))                   # 100 số lẻ phân biệt, p = 2
assert Solution().countDistinct(odd, 1, 2) == 100 * 101 // 2
print("tính chất OK")

tính chất OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đếm mảng con (không khử trùng)

In [21]:
def wrong_no_dedupe(nums, k, p):
    n = len(nums); ans = 0
    for i in range(n):
        cnt = 0
        for j in range(i, n):
            cnt += nums[j] % p == 0
            if cnt > k: break
            ans += 1                                   # SAI: đếm cả bản trùng
    return ans

print("bản sai :", wrong_no_dedupe([2, 3, 3, 2, 2], 2, 2), "| bản đúng:", Solution().countDistinct([2, 3, 3, 2, 2], 2, 2))

bản sai : 14 | bản đúng: 11


**Sửa:** khử trùng theo nội dung (set/trie/băm).

### Lỗi 2 — Nối số thành chuỗi **không có dấu phân cách**

In [22]:
def wrong_join(nums, k, p):
    n = len(nums); s = set()
    for i in range(n):
        cnt = 0; key = ""
        for j in range(i, n):
            cnt += nums[j] % p == 0
            if cnt > k: break
            key += str(nums[j])                         # SAI: [1, 23] và [12, 3] đều thành "123"
            s.add(key)
    return len(s)

nums = [1, 23, 12, 3]
print("bản sai :", wrong_join(nums, 4, 1), "| bản đúng:", Solution().countDistinct(nums, 4, 1))

bản sai : 9 | bản đúng: 10


**Sửa:** dùng `tuple`, hoặc chuỗi có dấu phân cách `",".join(...)` (và phân cách cả đầu/cuối nếu cần).

### Lỗi 3 — Nhầm điều kiện "chia hết": `p % x == 0`

In [23]:
def wrong_divisibility(nums, k, p):
    n = len(nums); s = set()
    for i in range(n):
        cnt = 0
        for j in range(i, n):
            cnt += p % nums[j] == 0                     # SAI: x là ước của p, không phải bội
            if cnt > k: break
            s.add(tuple(nums[i:j + 1]))
    return len(s)

print("bản sai :", wrong_divisibility([2, 3, 3, 2, 2], 2, 2), "| bản đúng:", Solution().countDistinct([2, 3, 3, 2, 2], 2, 2))
print("bản sai :", wrong_divisibility([1, 2, 3, 4], 1, 4), "| bản đúng:", Solution().countDistinct([1, 2, 3, 4], 1, 4))

bản sai : 11 | bản đúng: 11
bản sai : 6 | bản đúng: 10


Ví dụ 1 tình cờ cho cùng đáp án (với `p = 2` thì `2 % x == 0` ⇔ `x ∈ {1, 2}`, trùng với "x chẵn" trên mảng chỉ có 2 và 3); ví dụ thứ hai (`p = 4`) lộ lỗi.

**Sửa:** "`x` chia hết cho `p`" ⇔ `x % p == 0`.

### Lỗi 4 — Băm đơn với modulo nhỏ (va chạm)

In [24]:
def weak_hash(nums, k, p, B=10, P=97):
    n = len(nums); s = set()
    for i in range(n):
        h = cnt = 0
        for j in range(i, n):
            cnt += nums[j] % p == 0
            if cnt > k: break
            h = (h * B + nums[j]) % P                    # SAI: modulo quá nhỏ, B nhỏ hơn giá trị
            s.add(h)
    return len(s)

rng = random.Random(9)
nums = [rng.randint(1, 50) for _ in range(30)]
print("băm yếu:", weak_hash(nums, 30, 1), "| đúng:", Solution().countDistinct(nums, 30, 1))
print("ví dụ va chạm cấu trúc với B = 10: [1, 23] ->", (1 * 10 + 23), "và [3, 3] ->", 3 * 10 + 3)

băm yếu: 97 | đúng: 459
ví dụ va chạm cấu trúc với B = 10: [1, 23] -> 33 và [3, 3] -> 33


**Vì sao sai:** modulo 97 chỉ có 97 giá trị ⇒ hàng trăm dãy chắc chắn va chạm. Hơn nữa `B = 10 < 23` gây va chạm "cấu trúc" (`1·10 + 23 = 3·10 + 3`). **Sửa:** `B > max(nums)`, `P` nguyên tố lớn (~10⁹), băm kép; hoặc dùng trie (không va chạm).

### Lỗi 5 — Dừng sớm sai chỗ (`continue` thay vì `break`, hoặc `break` trước khi xét phần tử)

In [25]:
def wrong_continue(nums, k, p):
    n = len(nums); s = set()
    for i in range(n):
        cnt = 0
        for j in range(i, n):
            if nums[j] % p == 0: cnt += 1
            if cnt > k:
                cnt -= 1; continue                        # SAI: "bỏ qua" phần tử nhưng mảng con vẫn chứa nó
            s.add(tuple(nums[i:j + 1]))
    return len(s)

print("ví dụ 1  - bản sai :", wrong_continue([2, 3, 3, 2, 2], 2, 2), "| bản đúng:", Solution().countDistinct([2, 3, 3, 2, 2], 2, 2), "(trùng hợp: phần tử vi phạm ở cuối mảng)")
print("[2, 2, 1] - bản sai :", wrong_continue([2, 2, 1], 1, 2), "| bản đúng:", Solution().countDistinct([2, 2, 1], 1, 2))

ví dụ 1  - bản sai : 11 | bản đúng: 11 (trùng hợp: phần tử vi phạm ở cuối mảng)
[2, 2, 1] - bản sai : 4 | bản đúng: 3


Ở `[2, 2, 1]` với `k = 1`: từ `i = 0`, `j = 1` vi phạm, bản sai "bỏ qua" rồi ở `j = 2` thêm `[2, 2, 1]` — mảng có **hai** số chẵn.

**Vì sao sai:** "bỏ qua" phần tử vi phạm rồi đi tiếp vẫn thêm `nums[i:j+1]` chứa phần tử đó (hoặc nếu tự loại nó ra thì không còn là mảng **liên tiếp**). Khi `cnt > k`, mọi mảng con dài hơn từ cùng điểm đầu đều vi phạm ⇒ phải `break`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Số mảng con **phân biệt**" ⇒ cần khử trùng theo nội dung: `set(tuple)` (O(n³)), **trie** (O(n²) chính xác), **băm lăn** (O(n²) xác suất), hoặc suffix automaton/suffix array (cho n lớn).
- Điều kiện đơn điệu theo độ dài (đếm chỉ tăng) ⇒ dừng sớm.
- Khóa chuỗi: luôn có **dấu phân cách**.
- Băm: `B > giá trị lớn nhất`, `P` lớn, băm kép.
- Với n lớn hơn nhiều (10⁵), trie O(n²) không đủ; khi đó cần **suffix array + LCP** (đếm xâu con phân biệt = Σ(n − sa[i]) − Σ lcp) hoặc **suffix automaton** — đáng tìm hiểu sau bài này.
- Bài cùng họ: 1698 Number of Distinct Substrings in a String, 1316 Distinct Echo Substrings, 718 Maximum Length of Repeated Subarray, 2537.

## 15. Biến thể của bài

### 15.1 Biến thể A — Đếm mảng con phân biệt có **tổng ≤ S** (mọi phần tử dương)

Tổng cũng tăng theo `j` ⇒ cùng khung trie + dừng sớm.

In [26]:
def distinct_sum_at_most(nums, S):
    root = {}; ans = 0
    for i in range(len(nums)):
        node = root; tot = 0
        for j in range(i, len(nums)):
            tot += nums[j]
            if tot > S: break
            if nums[j] not in node:
                node[nums[j]] = {}; ans += 1
            node = node[nums[j]]
    return ans

def distinct_sum_brute(nums, S):
    return len({tuple(nums[i:j + 1]) for i in range(len(nums)) for j in range(i, len(nums)) if sum(nums[i:j + 1]) <= S})

print(distinct_sum_at_most([1, 2, 1, 2], 3), distinct_sum_brute([1, 2, 1, 2], 3))
rng = random.Random(151)
for _ in range(300):
    a = [rng.randint(1, 4) for _ in range(rng.randint(1, 12))]; S = rng.randint(1, 15)
    assert distinct_sum_at_most(a, S) == distinct_sum_brute(a, S)
print("khớp brute force")

4 4
khớp brute force


### 15.2 Biến thể B — Liệt kê (không chỉ đếm) các mảng con phân biệt hợp lệ, theo thứ tự từ điển

Duyệt trie theo thứ tự khóa tăng (DFS lặp để tránh đệ quy sâu — độ sâu tối đa n = 200, đệ quy cũng an toàn).

In [27]:
def list_distinct(nums, k, p):
    root = build_trie(nums, k, p); out = []; stack = [(root, [])]
    while stack:
        node, path = stack.pop()
        for x in sorted(node, reverse=True):
            stack.append((node[x], path + [x]))
        if path: out.append(path)
    return sorted(out)

print(list_distinct([2, 3, 3, 2, 2], 2, 2))

[[2], [2, 2], [2, 3], [2, 3, 3], [2, 3, 3, 2], [3], [3, 2], [3, 2, 2], [3, 3], [3, 3, 2], [3, 3, 2, 2]]


### 15.3 Biến thể C — Đếm **xâu con phân biệt** của chuỗi (LC 1698) bằng cùng kỹ thuật trie

In [28]:
def distinct_substrings(s):
    root = {}; ans = 0
    for i in range(len(s)):
        node = root
        for ch in s[i:]:
            if ch not in node:
                node[ch] = {}; ans += 1
            node = node[ch]
    return ans

print(distinct_substrings("aabbaba"), len({ "aabbaba"[i:j] for i in range(7) for j in range(i + 1, 8)}))

21 21


### 15.4 Biến thể D — Đếm mảng con phân biệt có **đúng k** phần tử chia hết cho p

Mỗi nội dung có một số lượng phần tử chia hết **cố định** ⇒ tập "≤ k" là hợp rời của "đúng 0", …, "đúng k" ⇒ `đúng(k) = ≤k − ≤(k−1)`.

In [29]:
def exactly_k(nums, k, p):
    at_most = lambda t: Solution().countDistinct(nums, t, p) if t >= 1 else trie_count(nums, 0, p)
    return at_most(k) - at_most(k - 1)

def exactly_k_brute(nums, k, p):
    return len({tuple(nums[i:j + 1]) for i in range(len(nums)) for j in range(i, len(nums))
                if sum(1 for x in nums[i:j + 1] if x % p == 0) == k})

print(exactly_k([2, 3, 3, 2, 2], 1, 2), exactly_k_brute([2, 3, 3, 2, 2], 1, 2))
rng = random.Random(154)
for _ in range(400):
    a = [rng.randint(1, 4) for _ in range(rng.randint(1, 12))]; k = rng.randint(1, len(a)); pp = rng.randint(1, 3)
    assert exactly_k(a, k, pp) == exactly_k_brute(a, k, pp)
print("đúng k = (<= k) - (<= k-1): khớp brute force")

5 5
đúng k = (<= k) - (<= k-1): khớp brute force


Lưu ý `trie_count(nums, 0, p)` vẫn chạy đúng với `k = 0` (ngoài ràng buộc đề) — hàm không giả định `k ≥ 1`.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi mảng con | đếm lại + tuple | O(n³) | O(n³) | kiểm chứng |
| 2. Đếm chạy + tuple | dừng sớm | O(n³) | O(n³) | n ≤ 200, code ngắn |
| 3. Trie | nút mới = mảng con mới | O(n²) | O(n²) | **mặc định**, chính xác |
| 4. Băm lăn kép | mã băm O(1) mỗi bước | O(n²) | O(n²) | ngôn ngữ không có dict lồng tiện |
| 5. Số lớn chính xác | cơ số B > max | O(n³/64) | O(n²) | Python, muốn chắc chắn không va chạm |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `countDistinct([1, 1, 1, 1], 2, 1)` và `countDistinct([3, 6, 9], 1, 3)`?

**Bài 2 (tính tay):** Với `[1, 2, 1, 2]`, `k = 1`, `p = 2`: liệt kê mảng con phân biệt hợp lệ.

**Bài 3 (điền TODO):** Viết `bt3_todo(nums, k, p)` trả về **độ dài lớn nhất** của một mảng con hợp lệ (không cần phân biệt) — O(n) bằng hai con trỏ.

**Bài 4 (tự viết lại):** Viết `bt4_todo(nums, k, p)` cài cách 4 (băm lăn kép) không nhìn code.

**Bài 5 (mini challenge):** Viết `bt5_todo(nums, k, p)` trả về số mảng con phân biệt hợp lệ **có độ dài đúng L** cho mọi `L` (list độ dài `n`, phần tử thứ `L−1` là số lượng) — dùng độ sâu của nút trong trie.

In [30]:
def bt3_todo(nums, k, p):
    # TODO: độ dài lớn nhất của mảng con có <= k phần tử chia hết cho p
    pass

def bt4_todo(nums, k, p):
    # TODO: băm lăn kép
    pass

def bt5_todo(nums, k, p):
    # TODO: số mảng con phân biệt theo từng độ dài
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: `p = 1` ⇒ mọi phần tử chia hết. Mức 2: chỉ mảng con dài ≤ 2, nội dung chỉ có `[1]`, `[1, 1]`. Mức 3: `2`; ví dụ hai: mọi phần tử chia hết cho 3, `k = 1` ⇒ chỉ độ dài 1 ⇒ `3`.

**Bài 2.** Mức 1: phần tử chia hết cho 2: các số 2. Mức 2: mỗi mảng con chứa tối đa một số 2. Mức 3: `[1], [2], [1,2], [2,1], [1,2,1]` ⇒ 5.

**Bài 3.** Mức 1: cửa sổ `[l, r]` với `cnt ≤ k`. Mức 2: khi `cnt > k`, tăng `l` tới khi bỏ được một phần tử chia hết. Mức 3: lấy max `r − l + 1`.

**Bài 4.** Mức 1: hai cặp `(B, P)`. Mức 2: `h = (h·B + x) % P`. Mức 3: thêm cặp `(h1, h2)` vào `set`.

**Bài 5.** Mức 1: mỗi nút trie ở độ sâu `d` ứng với mảng con dài `d`. Mức 2: khi tạo nút ở bước `j` của điểm đầu `i`, độ sâu là `j − i + 1`. Mức 3: `res[j − i] += 1`.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `2`, `3`. **Bài 2:** 5 mảng con: `[1], [2], [1,2], [2,1], [1,2,1]`.

In [31]:
assert Solution().countDistinct([1, 1, 1, 1], 2, 1) == 2
assert Solution().countDistinct([3, 6, 9], 1, 3) == 3
assert sorted(list_distinct([1, 2, 1, 2], 1, 2)) == sorted([[1], [2], [1, 2], [2, 1], [1, 2, 1]])

def bt3(nums, k, p):
    l = cnt = best = 0
    for r, x in enumerate(nums):
        cnt += x % p == 0
        while cnt > k:
            cnt -= nums[l] % p == 0; l += 1
        best = max(best, r - l + 1)
    return best

def bt4(nums, k, p):
    return double_hash(nums, k, p)

def bt5(nums, k, p):
    n = len(nums); root = {}; res = [0] * n
    for i in range(n):
        node = root; cnt = 0
        for j in range(i, n):
            cnt += nums[j] % p == 0
            if cnt > k: break
            if nums[j] not in node:
                node[nums[j]] = {}; res[j - i] += 1
            node = node[nums[j]]
    return res

rng = random.Random(1961)
for _ in range(600):
    nums = [rng.randint(1, 4) for _ in range(rng.randint(1, 15))]; k = rng.randint(1, len(nums)); p = rng.randint(1, 4)
    subs = {tuple(nums[i:j + 1]) for i in range(len(nums)) for j in range(i, len(nums))
            if sum(1 for x in nums[i:j + 1] if x % p == 0) <= k}
    assert bt3(nums, k, p) == max(len(t) for t in subs) if subs else bt3(nums, k, p) == 0
    assert bt4(nums, k, p) == len(subs)
    res = bt5(nums, k, p)
    assert sum(res) == len(subs) and all(res[L - 1] == sum(1 for t in subs if len(t) == L) for L in range(1, len(nums) + 1))
print("Đáp án bài tập: tất cả assert OK")

Đáp án bài tập: tất cả assert OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
root = {}; ans = 0
for i in range(n):
    node = root; cnt = 0
    for j in range(i, n):
        cnt += nums[j] % p == 0
        if cnt > k: break                 # đơn điệu -> dừng
        if nums[j] not in node:           # mảng con nums[i..j] mới
            node[nums[j]] = {}; ans += 1
        node = node[nums[j]]
return ans
```

**Checklist:**
- [ ] Khử trùng theo **nội dung** (trie / set tuple / băm tốt).
- [ ] `x % p == 0` (x là bội của p).
- [ ] `break` khi `cnt > k`.
- [ ] Không nối số thành chuỗi thiếu dấu phân cách.
- [ ] Nếu băm: `B > max`, `P` lớn, băm kép.